<table dir="ltr" width="100%"><tr><td width="50%" valign="top" dir="ltr" lang="en" align="left"><h1>Masar · Day 1</h1><p>Meaad Al-Marri · SDA-DSC-214<br>One notebook, Labs 01 and 02. Run in order. This is the first stage of the final project.</p></td><td width="50%" valign="top" dir="rtl" lang="ar" align="right"><h1>مسار · اليوم الأول</h1><p>ميعاد المري · SDA-DSC-214<br>دفتر واحد للابين 01 و02. شغله بالترتيب؛ هذه المرحلة الأولى من المشروع النهائي.</p></td></tr></table>



<table dir="ltr" width="100%"><tr><td width="50%" valign="top" dir="ltr" lang="en" align="left"><h2>Setup</h2><p>Colab: use CPU, save a copy and run this cell. Local Jupyter: install requirements-day01.txt first. No Docker, API token or GPU.</p></td><td width="50%" valign="top" dir="rtl" lang="ar" align="right"><h2>الإعداد</h2><p>في Colab اختر CPU واحفظ نسخة وشغل هذه الخلية. في Jupyter المحلي ثبت requirements-day01.txt أولًا. لا يلزم Docker أو رمز API أو GPU.</p></td></tr></table>



In [1]:
from pathlib import Path
import os, sys, subprocess, importlib.metadata
IS_COLAB = Path('/content').is_dir() and 'google.colab' in sys.modules
if IS_COLAB:
    ROOT = Path('/content/masar-modern-data-engineering')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/almiyead-rgb/masar-modern-data-engineering.git',str(ROOT)], check=True)
    pins = {'pyspark':'3.5.8','delta-spark':'3.3.3','py4j':'0.10.9.9'}
    missing = []
    for package, version in pins.items():
        try: observed = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: observed = None
        if observed != version: missing.append(f'{package}=={version}')
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
    candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    if not any((p/'bin/java').is_file() for p in candidates):
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
        candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    os.environ['JAVA_HOME'] = str(next(p for p in candidates if (p/'bin/java').is_file()))
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.json').is_file()),None)
    if ROOT is None: raise FileNotFoundError('Open from the repository root; see docs/SETUP.md')
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])


Repository: /tmp/masar_course_1_m1jsz92j/course
Python: 3.11.16


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 01 preparation · Inspect the fixed sources</h1><p>This is the source-inspection part, not the complete Bronze lab. The complete lab also requires real Delta writes and replay evidence.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>تحضير اللاب 01 · فحص المصادر الثابتة</h1><p>هذا جزء فحص المصادر لا لاب Bronze كاملًا؛ فاللاب الكامل يتطلب كتابة Delta فعلية وأدلة الإعادة.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal and setup</h2><p>Read the manifest, count the three base feeds and examine keys, relationships and city-label formatting. Keep the whole repository together; shared code is in <code>src/masar/sources.py</code>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف والإعداد</h2><p>اقرأ السجل وعد المصادر الثلاثة وافحص المفاتيح والعلاقات وكتابة المدن. احتفظ بالمستودع كاملًا؛ الكود المشترك في <code dir="ltr">src/masar/sources.py</code>.</p></td></tr></tbody></table>

In [2]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Verify and load</h2><p>Reject changed source files before interpreting their content. This does not modify the inputs.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>1. التحقق والتحميل</h2><p>ارفض تغير ملفات المصدر قبل تفسير محتواها. لا تغير الخطوة المدخلات.</p></td></tr></tbody></table>

In [3]:
from masar.sources import verify_manifest, load_sources, profile_sources
SOURCE = ROOT / "data" / "masar-small-v1"
manifest = verify_manifest(SOURCE)
feeds = load_sources(SOURCE)
print("Dataset:", manifest["label"])
print("Verified manifest files:", len(manifest["files"]))
print(json.dumps({name: len(rows) for name, rows in feeds.items()}, indent=2))
print("Trip columns:", list(feeds["trips"][0]))
print("First synthetic event:", json.dumps(feeds["gps_events"][0], ensure_ascii=False))

Dataset: MASAR_SMALL_V1
Verified manifest files: 10
{
  "trips": 72,
  "drivers": 6,
  "gps_events": 216
}
Trip columns: ['trip_id', 'driver_id', 'city', 'start_ts', 'end_ts', 'fare_sar', 'distance_km']
First synthetic event: {"city": "Riyadh", "event_id": "SYN_E0001_0", "event_ts": "2026-06-01T06:00:00+03:00", "location": {"lat": 24.7, "lon": 46.7}, "synthetic": true, "trip_id": "SYN_T0001"}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Profile before cleaning</h2><p>Count keys and inspect relationships. The city-normalization comparison is a view of the inputs, not a rewrite of Bronze.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>2. افحص قبل التنظيف</h2><p>عد المفاتيح وافحص العلاقات. مقارنة توحيد المدن فحص للمدخلات، وليست إعادة كتابة Bronze.</p></td></tr></tbody></table>

In [4]:
result = profile_sources(SOURCE)
print(json.dumps(result["profile"], indent=2))
print(json.dumps(result["relations"], indent=2))
print(json.dumps(result["city_profile"], indent=2))

{
  "trips": {
    "rows": 72,
    "key": "trip_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "distance_km": 0,
      "driver_id": 0,
      "end_ts": 0,
      "fare_sar": 0,
      "start_ts": 0,
      "trip_id": 0
    }
  },
  "drivers": {
    "rows": 6,
    "key": "driver_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "driver_id": 0,
      "driver_rating": 0,
      "vehicle_type": 0
    }
  },
  "gps_events": {
    "rows": 216,
    "key": "event_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "event_id": 0,
      "event_ts": 0,
      "location": 0,
      "synthetic": 0,
      "trip_id": 0
    }
  }
}
{
  "trips_without_driver": 0,
  "events_without_trip": 0,
  "events_with_invalid_coordinates": 0
}
{
  "original_labels": {
    " dammam ": 3,
    " jeddah ": 3,
    " riyad

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Check and preserve evidence</h2><p>The checks below concern the source files only. A replay into append-only Bronze is a separate operation and has not been run here.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>3. تحقق واحفظ الأدلة</h2><p>تخص الفحوص التالية ملفات المصدر فقط. إعادة الاستيعاب في Bronze بنمط الإضافة عملية منفصلة لم تنفذ هنا.</p></td></tr></tbody></table>

In [5]:
if not all(result["checks"].values()):
    raise AssertionError(result["checks"])
output = RUN / "source_inspection.json"
output.write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print(json.dumps(result["checks"], indent=2))
print("PASS: source inspection only")
print("Saved:", output.name)
print("Source inspection complete. Continue to the Bronze section.")

{
  "source_counts": true,
  "unique_base_keys": true,
  "base_top_level_complete": true,
  "valid_links_and_coordinates": true,
  "three_events_per_trip": true,
  "base_city_set": true
}
PASS: source inspection only
Saved: source_inspection.json
Source inspection complete. Continue to the Bronze section.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Interpretation and next step</h2><p>Explain why joining each trip to all its events changes the row grain. Explain why a unique base key does not guarantee uniqueness after replay. Return to the Lab 01 contract; its engine steps remain pending.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>التفسير والخطوة التالية</h2><p>اشرح لماذا يغير ربط الرحلة بجميع أحداثها مستوى الصف، ولماذا لا يضمن المفتاح الفريد في المصدر عدم التكرار بعد الإعادة. ارجع إلى مواصفات اللاب 01؛ خطوات محركه ما تزال معلقة.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 01 · Real Delta Bronze</h1><p>SDA-DSC-214 · Meaad Al-Marri</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اللاب 01 · Bronze فعلية باستخدام Delta</h1><p>SDA-DSC-214 · ميعاد المري</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Setup · report the actual environment</h2><p>Only inspect dependencies here. This cell does not install packages or start Spark.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الإعداد · اعرض البيئة الفعلية</h2><p>افحص المتطلبات فقط هنا. لا تثبت هذه الخلية حزمًا ولا تبدأ Spark.</p></td></tr></tbody></table>

In [6]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.16",
  "java": "openjdk version \"17.0.20.1\" 2026-08-18",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Start a clean, bounded run</h2><p>Fail clearly if dependencies are missing; never switch engines silently. Existing work is retained.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>ابدأ تشغيلًا جديدًا محدود النطاق</h2><p>توقف بوضوح عند نقص المتطلبات دون تبديل المحرك بصمت. تبقى الأعمال السابقة محفوظة.</p></td></tr></tbody></table>

In [7]:
require_environment()
from masar.workspace import new_workspace, require_fixed_dataset, write_json, workspace_path
require_fixed_dataset(SOURCE)
WORK = new_workspace(ROOT, "day01_bronze")
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_g_589daf


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Read one source</h2><p>CSV strings remain unchanged. The count action starts computation.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>اقرأ مصدرًا واحدًا</h2><p>تبقى نصوص CSV كما هي. يطلق فعل العد الحساب الفعلي.</p></td></tr></tbody></table>

In [8]:
from masar.bronze import raw_frame, ingest_feed, verify_bronze
# Functions are imported without starting Spark. The next cell executes the lab.
print("Bronze functions loaded. The next cell writes and reads the Delta tables.")

Bronze functions loaded. The next cell writes and reads the Delta tables.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Load, replay, verify and retain evidence</h2><p>The <code>try/finally</code> always stops the session after this block, even on failure. Reuse of a committed batch id is rejected. See <a href="../src/masar/bronze.py">shared source</a>; this is not a placeholder or mock engine.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>استوعب وأعد الوصول وتحقق واحفظ الدليل</h2><p>توقف <code>try/finally</code> الجلسة بعد هذه الكتلة حتى عند الفشل. تُرفض إعادة استخدام معرف دفعة محفوظ. راجع <a href="../src/masar/bronze.py">المصدر المشترك</a>؛ ليست هذه محاكاة للمحرك أو موضعًا فارغًا.</p></td></tr></tbody></table>

In [9]:
from masar.workspace import record_bronze_success
spark = start_spark(WORK)
try:
    print("Spark:", spark.version)
    raw_trips = raw_frame(spark, SOURCE, "trips")
    raw_trips.printSchema()
    raw_trips.show(3, truncate=False)
    print("Source trips:", raw_trips.count())
    for feed in ("trips", "drivers", "gps_events"):
        print(ingest_feed(spark, SOURCE, WORK, feed, "base_001"))
    print(ingest_feed(spark, SOURCE, WORK, "trips", "replay_002"))
    report = verify_bronze(spark, SOURCE, WORK)
    record_bronze_success(ROOT, WORK)
    print(json.dumps({"counts": report["counts"], "checks": report["checks"]}, indent=2))
    print("Retained:", (WORK / "reports/bronze.json").relative_to(ROOT))
finally:
    spark.stop()

https://repo.maven.apache.org/maven2 added as a remote repository with the name: repo-1


:: loading settings :: url = jar:file:/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/runner/.ivy2/cache
The jars for the packages stored in: /home/runner/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f334b0ae-e56e-4cb0-b68a-7bbb3a8a539f;1.0
	confs: [default]


	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central


	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.3.3/delta-spark_2.12-3.3.3.jar ...


...

{'feed': 'trips', 'batch_id': 'base_001', 'appended_rows': 72, 'total_rows': 72, 'version': 0}


{'feed': 'drivers', 'batch_id': 'base_001', 'appended_rows': 6, 'total_rows': 6, 'version': 0}


{'feed': 'gps_events', 'batch_id': 'base_001', 'appended_rows': 216, 'total_rows': 216, 'version': 0}


{'feed': 'trips', 'batch_id': 'replay_002', 'appended_rows': 72, 'total_rows': 144, 'version': 1}


{
  "counts": {
    "trips": 144,
    "drivers": 6,
    "gps_events": 216
  },
  "checks": {
    "expected_totals": true,
    "two_trip_deliveries": true,
    "business_trip_count_unchanged": true,
    "initial_snapshot_retained": true,
    "base_csv_columns_are_strings": true,
    "payloads_and_source_hashes_preserved": true,
    "real_delta_files_present": true
  }
}
Retained: outputs/day01_bronze_g_589daf/reports/bronze.json


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Checks and next step</h2><p>A successful scenario has 144 Bronze trip deliveries but 72 distinct trip ids. Read <a href="PRACTICE.md">the reasoning prompts</a>, complete Lab 01 notes, then use the same successful workspace in <a href="STUDENT.ipynb">Lab 02</a>. No generated table is automatically committed to Git.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الفحوص والخطوة التالية</h2><p>يحتوي السيناريو الناجح 144 سجل وصول في Bronze و72 معرف رحلة فريدًا. اقرأ <a href="PRACTICE.md">أسئلة التفكير</a> وأكمل ملاحظات اللاب 01، ثم استخدم مساحة العمل الناجحة نفسها في <a href="STUDENT.ipynb">اللاب 02</a>. لا يضاف أي جدول مولد إلى Git تلقائيًا.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 02 preparation · Cost arithmetic</h1><p>Compare two operation policies using hypothetical teaching units (TU). This is not a quotation, currency forecast or Spark runtime benchmark.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>تحضير اللاب 02 · حساب التكلفة</h1><p>قارن سياستي تشغيل بوحدات تعليم افتراضية TU. هذا ليس عرض سعر أو توقع عملة أو قياسًا لأداء Spark.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal and setup</h2><p>Show how work hours, startup and overhead affect the result; do not assume scheduled compute is always cheaper. Shared code is in <code>src/masar/cost.py</code>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف والإعداد</h2><p>بيّن أثر ساعات العمل والبدء والتكاليف الإضافية؛ ولا تفترض أن الحوسبة المجدولة أرخص دائمًا. الكود المشترك في <code dir="ltr">src/masar/cost.py</code>.</p></td></tr></tbody></table>

In [10]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Inspect assumptions</h2><p>The base horizon is one 30-day teaching month. The storage charge is a monthly assumption; keep the horizon fixed when interpreting this exercise.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>1. افحص الافتراضات</h2><p>الفترة الأساسية شهر تعليمي من 30 يومًا. تكلفة التخزين افتراض شهري؛ حافظ على هذه الفترة عند تفسير التمرين.</p></td></tr></tbody></table>

In [11]:
from decimal import Decimal
from masar.cost import DEFAULTS, evaluate_cost, serializable, cost_report
print(json.dumps(DEFAULTS, indent=2))
result = cost_report()
print(json.dumps(result["base"], indent=2))

{
  "days": "30",
  "hours_per_day": "24",
  "cores": "4",
  "price_per_core_hour": "0.50",
  "storage_gib": "100",
  "storage_price_per_gib_month": "0.20",
  "work_hours_per_day": "2",
  "startup_hours_per_day": "0.25",
  "scheduled_extra_monthly": "30",
  "always_on_extra_monthly": "0"
}
{
  "unit": "TU (hypothetical teaching units; not currency)",
  "always_on_hours": "720",
  "scheduled_hours": "67.50",
  "always_on_compute": "1440.00",
  "scheduled_compute": "135.0000",
  "storage_each": "20.00",
  "always_on_total": "1460.00",
  "scheduled_total": "185.0000",
  "difference": "1275.0000",
  "reduction_fraction": "0.8732876712328767123287671233",
  "break_even_work_hours_per_day": "23.25"
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Test workload sensitivity</h2><p>Only work hours change below. The remaining assumptions stay fixed so the comparison is interpretable.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>2. اختبر حساسية حمل العمل</h2><p>تتغير ساعات العمل فقط أدناه؛ وتبقى بقية الافتراضات ثابتة لتكون المقارنة قابلة للتفسير.</p></td></tr></tbody></table>

In [12]:
print("Work h/day | Always-on TU | Scheduled TU | Difference TU")
for row in result["sensitivity"]:
    print(f"{row['work_hours_per_day']:>10} | {row['always_on_total']:>12} | {row['scheduled_total']:>12} | {row['difference']:>13}")

Work h/day | Always-on TU | Scheduled TU | Difference TU
         0 |      1460.00 |        50.00 |       1410.00
         1 |      1460.00 |     125.0000 |     1335.0000
         2 |      1460.00 |     185.0000 |     1275.0000
         4 |      1460.00 |     305.0000 |     1155.0000
         8 |      1460.00 |     545.0000 |      915.0000
        12 |      1460.00 |     785.0000 |      675.0000
        18 |      1460.00 |    1145.0000 |      315.0000
        23 |      1460.00 |    1445.0000 |       15.0000
     23.25 |      1460.00 |    1460.0000 |        0.0000
     23.75 |      1460.00 |    1490.0000 |      -30.0000


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Check boundaries and invalid inputs</h2><p>Test equality, a counterexample, zero rates and a rejected invalid input. Passing arithmetic does not establish cloud economics or real performance.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>3. افحص الحدود والمدخلات غير الصالحة</h2><p>اختبر التعادل والمثال المضاد والمعدل الصفري ورفض مدخل غير صالح. نجاح الحساب لا يثبت اقتصاديات سحابية أو أداءً فعليًا.</p></td></tr></tbody></table>

In [13]:
base = evaluate_cost(DEFAULTS)
checks = {
    "base_totals": (base["always_on_total"], base["scheduled_total"]) == (Decimal("1460"), Decimal("185")),
    "break_even": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.25"})["difference"] == 0,
    "counterexample": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.75"})["difference"] < 0,
    "zero_rate": evaluate_cost({**DEFAULTS, "price_per_core_hour": "0"})["break_even_work_hours_per_day"] is None,
}
try:
    evaluate_cost({**DEFAULTS, "cores": "-1"})
except ValueError:
    checks["negative_input_rejected"] = True
else:
    checks["negative_input_rejected"] = False
if not all(checks.values()):
    raise AssertionError(checks)
result["checks"] = checks
print(json.dumps(checks, indent=2))

{
  "base_totals": true,
  "break_even": true,
  "counterexample": true,
  "zero_rate": true,
  "negative_input_rejected": true
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>4. Save evidence</h2><p>The artifact explicitly records exclusions. Keep real Spark measurements separate when the full lab is available.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>4. احفظ الأدلة</h2><p>يسجل الملف ما يستبعده الحساب صراحة. افصل قياسات Spark الفعلية عندما يتاح اللاب الكامل.</p></td></tr></tbody></table>

In [14]:
output = RUN / "cost_model_result.json"
output.write_text(json.dumps(result, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("PASS: hypothetical cost arithmetic only")
print("Saved:", output.name)
print("Cost arithmetic complete. Continue to the measured Spark comparison.")

PASS: hypothetical cost arithmetic only
Saved: cost_model_result.json
Cost arithmetic complete. Continue to the measured Spark comparison.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Interpretation and next step</h2><p>Explain the break-even workload and one excluded cost. Do not label TU as SAR or claim a measured speed-up. Continue with the Lab 02 contract when the actual benchmark is verified.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>التفسير والخطوة التالية</h2><p>اشرح حمل التعادل وتكلفة مستبعدة واحدة. لا تسم وحدات TU ريالات ولا تدع تسارعًا مقاسًا. أكمل مواصفات اللاب 02 عند التحقق من القياس الفعلي.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 02 · Observe actual Spark scans</h1><p>SDA-DSC-214 · Meaad Al-Marri</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اللاب 02 · لاحظ قياس Spark الفعلي</h1><p>SDA-DSC-214 · ميعاد المري</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Setup</h2><p>Inspect the actual environment before accessing a prior workspace.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الإعداد</h2><p>افحص البيئة الفعلية قبل الوصول إلى مساحة عمل سابقة.</p></td></tr></tbody></table>

In [15]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.16",
  "java": "openjdk version \"17.0.20.1\" 2026-08-18",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Reuse the successful project state</h2><p>No source regeneration, table overwrite or new independent project.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>استخدم حالة المشروع الناجحة</h2><p>لا إعادة توليد للمصدر ولا استبدال للجداول ولا مشروع جديد مستقل.</p></td></tr></tbody></table>

In [16]:
require_environment()
from masar.workspace import completed_bronze_workspace, require_fixed_dataset
require_fixed_dataset(SOURCE)
WORK = completed_bronze_workspace(ROOT)
spark = start_spark(WORK)
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_g_589daf


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Measure equal-result actions</h2><p>CSV versus Delta version 0; one warm-up each, four measurements each in balanced order. Shared code retains the raw measurements and actual query plans. Session startup and ingestion are outside the measured interval.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>قس أفعالًا متساوية النتائج</h2><p>CSV مقابل نسخة Delta رقم صفر؛ تهيئة لكل مسار وأربعة قياسات لكل مسار بترتيب متوازن. يحفظ الكود القياسات الخام وخطط الاستعلام الفعلية. إقلاع الجلسة والاستيعاب خارج الفترة المقاسة.</p></td></tr></tbody></table>

In [17]:
from masar.benchmark import benchmark
try:
    report = benchmark(spark, SOURCE, WORK, repetitions=4)
    print(json.dumps(report["expected_and_observed_aggregate"], indent=2))
    print(json.dumps(report["measurements"], indent=2))
    print("Plans:", report["plans"])
    print("Evidence:", (WORK / "reports/benchmark.json").relative_to(ROOT))
finally:
    spark.stop()

{
  "rows": 72,
  "nonnull_fares": 72,
  "fare_total": "1794.60"
}
{
  "csv": {
    "samples_s": [
      0.08258816699998306,
      0.08171742300001483,
      0.10791064699998287,
      0.11294147699999257
    ],
    "median_s": 0.09524940699998297,
    "min_s": 0.08171742300001483,
    "max_s": 0.11294147699999257
  },
  "delta_v0": {
    "samples_s": [
      0.5821868119999749,
      0.5518524470000159,
      1.0129817159999845,
      0.8470437240000024
    ],
    "median_s": 0.7146152679999886,
    "min_s": 0.5518524470000159,
    "max_s": 1.0129817159999845
  }
}
Plans: {'csv': 'reports/plans/csv.txt', 'delta_v0': 'reports/plans/delta_v0.txt'}
Evidence: outputs/day01_bronze_g_589daf/reports/benchmark.json


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Interpret and retain</h2><p>Keep all samples and explain variability. Repeated reads may use OS/JVM/metadata caches; this is not a cold-cache or production test. Complete <a href="../templates/BENCHMARKS.md">BENCHMARKS.md</a> and Lab 02 notes. See <a href="COMPLETION.md">the Day 1 handoff</a>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>فسر واحتفظ</h2><p>احفظ جميع العينات وفسر التفاوت. قد تستفيد القراءات المتكررة من ذاكرة نظام التشغيل وJVM والبيانات الوصفية؛ ليست تجربة ذاكرة فارغة أو اختبار إنتاج. أكمل <a href="../templates/BENCHMARKS.md">BENCHMARKS.md</a> وملاحظات اللاب 02. راجع <a href="COMPLETION.md">تسليم اليوم الأول</a>.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><tr><td width="50%" valign="top" dir="ltr" lang="en" align="left"><h2>Save the handoff</h2><p>Keep the Delta tables, reports and executed notebook for Day 2. Complete the learning notes linked from project/SUBMISSION.md; there is no extra final project.</p></td><td width="50%" valign="top" dir="rtl" lang="ar" align="right"><h2>احفظ مخرجات الانتقال</h2><p>احتفظ بجداول Delta والتقارير والدفتر المنفذ لليوم الثاني. أكمل ملاحظات التعلم في project/SUBMISSION.md؛ لا يوجد مشروع نهائي إضافي.</p></td></tr></table>



In [18]:
# DAY01_HANDOFF_V2: retain the pointer and all small reports as well as Delta files.
from pathlib import Path
import zipfile
from masar.workspace import completed_bronze_workspace
WORK = completed_bronze_workspace(ROOT)
pointer = ROOT / 'outputs/day01_bronze_success.json'
files_to_save = {pointer, *(p for p in WORK.rglob('*') if p.is_file())}
for name in ('source_inspection.json', 'cost_model_result.json'):
    files_to_save.update((ROOT / 'outputs').rglob(name))
archive = ROOT / 'outputs/day01_handoff.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(files_to_save):
        bundle.write(path, arcname=path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    assert bundle.read('outputs/day01_bronze_success.json') == pointer.read_bytes()
    assert any(name.endswith('source_inspection.json') for name in bundle.namelist())
    assert any(name.endswith('cost_model_result.json') for name in bundle.namelist())
print('Keep this ZIP for the next day:', archive)
print('Also save this notebook with outputs and your LAB01/LAB02 notes.')
if IS_COLAB:
    from google.colab import files
    files.download(str(archive))

Keep this ZIP for the next day: /tmp/masar_course_1_m1jsz92j/course/outputs/day01_handoff.zip
Also save this notebook with outputs and your LAB01/LAB02 notes.
